<a href="https://colab.research.google.com/github/p-tech/ntu-growth-hacking/blob/main/blog-clustering/Oddbox_Blog_Topic_Modelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Oddbox blog topic modelling and content audit

**NTU Growth Hacking | Google Colab**

What does the first page of the Oddbox blog cover, and which questions raised in customer reviews could its articles answer?

Run the cells in order. You will collect the latest articles on the first blog listing page, discover and name topics, explore links between topics in 3D, check review questions against article evidence, and categorise a draft. You can change the topic count and compare the results. The sample is a snapshot of one listing page and can change between runs.

In [1]:
%pip -q install beautifulsoup4 wordcloud plotly python-docx

In [2]:
from io import BytesIO
from itertools import combinations
from pathlib import Path
from urllib.parse import urljoin, urlsplit
from time import sleep
import re
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import requests
from bs4 import BeautifulSoup
from docx import Document
from IPython.display import display
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import NMF, PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
from wordcloud import WordCloud

## 1. Collect the first page of articles

The notebook uses the main **Our Blog** listing. It requests each article linked from that listing and checks the extracted title, body and word count. If the layout changes, you can paste the article URLs from that page into `ARTICLE_URLS`. If access fails, upload a prepared CSV with `title`, `url` and `text` columns and an optional `category` column. Set `USE_CSV_UPLOAD` to `True` first. Only collect pages you have permission to use.

In [ ]:
BLOG_URL = "https://www.oddbox.co.uk/blog"
ARTICLE_URLS = []
USE_CSV_UPLOAD = False

def read_upload():
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Choose one file.")
    return next(iter(uploaded.items()))

def get_page(url):
    response = requests.get(url, timeout=20)
    response.raise_for_status()
    return BeautifulSoup(response.content, "html.parser")

def listing_urls(page):
    urls = []
    for link in page.select("main article a.curated-posts-section__post-link[href]"):
        path = urlsplit(link["href"]).path.rstrip("/")
        if re.fullmatch(r"/blog/[a-zA-Z0-9_-]+", path):
            urls.append(urljoin(BLOG_URL, path))
    if not urls:
        raise ValueError("No article cards found. Paste first-page article URLs or use CSV upload.")
    return list(dict.fromkeys(urls))

def extract_article(url, page):
    article = page.select_one("article.post")
    if article is None or article.select_one("h1") is None:
        raise ValueError("Article title or body could not be identified.")
    title = article.select_one("h1").get_text(" ", strip=True)
    category = page.select_one('meta[property="article:section"]')
    category_name = category.get("content", "").strip() if category else ""
    if not category_name:
        category_link = article.select_one('a[href*="/blog/categories/"]')
        category_name = category_link.get_text(" ", strip=True) if category_link else ""
    body_parts = article.select(
        ".post__description, .richtext-section h2, .richtext-section h3, "
        ".richtext-section h4, .richtext-section p, .richtext-section li"
    )
    paragraphs = []
    for part in body_parts:
        if part.find_parent(["li", "p"]):
            continue
        paragraph = part.get_text(" ", strip=True)
        if paragraph and paragraph not in paragraphs:
            paragraphs.append(paragraph)
    return {"title": title, "url": url, "text": "\n\n".join(paragraphs),
            "category": category_name}

if USE_CSV_UPLOAD:
    _, payload = read_upload()
    articles = pd.read_csv(BytesIO(payload), keep_default_na=False)
    if not {"title", "url", "text"}.issubset(articles.columns):
        raise ValueError("The CSV needs title, url and text columns.")
    articles = articles[[column for column in ["title", "url", "text", "category"]
                         if column in articles.columns]].copy()
else:
    urls = list(dict.fromkeys(ARTICLE_URLS)) or listing_urls(get_page(BLOG_URL))
    rows, failures = [], []
    print(f"Found {len(urls)} article links.")
    for url in urls:
        try:
            rows.append(extract_article(url, get_page(url)))
        except (requests.RequestException, ValueError) as error:
            failures.append({"url": url, "issue": str(error)})
        sleep(0.35)
    if failures:
        display(pd.DataFrame(failures))
    articles = pd.DataFrame(rows, columns=["title", "url", "text", "category"])

if "category" not in articles:
    articles["category"] = ""
articles["category"] = articles["category"].fillna("").astype(str).str.strip()
articles["words"] = articles["text"].astype(str).str.split().str.len()
short = articles[articles["words"] < 50]
if not short.empty:
    print("Excluded because fewer than 50 words were extracted:")
    display(short[["title", "url", "words"]])
articles = (articles[articles["words"] >= 50]
            .drop_duplicates("url").drop_duplicates("text").reset_index(drop=True))
articles.insert(0, "article_id", [f"A{i:02d}" for i in range(1, len(articles)+1)])
display(articles[["article_id", "title", "category", "words", "url"]])
if len(articles) < 3:
    raise ValueError("Collect at least three usable articles. Check the extraction or use CSV upload.")
articles.to_csv("blog_articles.csv", index=False)

## 2. Discover the topics

**TF-IDF** gives more weight to terms that help distinguish articles. **N-grams** let the model use single words and two-word phrases such as *food waste*. **NMF** looks for sets of terms that recur together and gives every article a score for each topic. A single article can cover several topics.

Start with four topics. Compare the charts and passages after trying three or five. Keep the result that gives you distinct, interpretable themes. The normalised shares describe the mix of topics found by this model. Treat them as descriptive scores.

In [ ]:
TOPIC_COUNT = 4
EXTRA_STOP_WORDS = ["oddbox"]
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

vectoriser = TfidfVectorizer(
    stop_words=sorted(set(ENGLISH_STOP_WORDS).union(EXTRA_STOP_WORDS)),
    ngram_range=(1, 2), min_df=2, max_df=0.9
)
matrix = vectoriser.fit_transform(articles["text"])
k = min(TOPIC_COUNT, len(articles) - 1, matrix.shape[1])
if k < 2:
    raise ValueError("Choose at least two topics and check for a varied set of articles.")
model = NMF(n_components=k, init="nndsvda", random_state=42, max_iter=1500)
weights = model.fit_transform(matrix)
terms = vectoriser.get_feature_names_out()
topic_ids = [f"T{i+1}" for i in range(k)]

def topic_shares(values):
    # Put NMF components on the same scale before comparing their contribution.
    scaled = values * model.components_.sum(axis=1)
    totals = scaled.sum(axis=1, keepdims=True)
    return np.divide(scaled, totals, out=np.zeros_like(scaled), where=totals > 0)

shares = topic_shares(weights)
topic_terms = []
for index, component in enumerate(model.components_):
    top = component.argsort()[::-1][:18]
    frequencies = {terms[j]: component[j] for j in top if component[j] > 0}
    topic_terms.append(frequencies)
    if not frequencies:
        print(f"{topic_ids[index]} has no terms. Try fewer topics.")
        continue
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    axes[0].imshow(WordCloud(width=650, height=300, background_color="white",
                             random_state=42).generate_from_frequencies(frequencies))
    axes[0].axis("off")
    chart = list(frequencies.items())[:10][::-1]
    axes[1].barh([phrase for phrase, _ in chart], [weight for _, weight in chart])
    axes[1].set_xlabel("Topic weight")
    fig.suptitle(topic_ids[index])
    plt.tight_layout()
    plt.show()
    for row in shares[:, index].argsort()[::-1][:2]:
        if shares[row, index] <= 0:
            continue
        paragraphs = articles.loc[row, "text"].split("\n\n")
        scores = topic_shares(model.transform(vectoriser.transform(paragraphs)))
        # Prefer passages with both a strong topic match and relevant words.
        evidence = cosine_similarity(vectoriser.transform(paragraphs),
                                     model.components_[index:index+1]).ravel()
        passage = paragraphs[(scores[:, index] * evidence).argmax()]
        print(f"{articles.loc[row, 'article_id']} | {articles.loc[row, 'title']} "
              f"| {shares[row, index]:.0%} of modelled topic weight")
        print(passage[:420], "\n")

Name each topic using its phrases and example passages. Use short labels such as *Storage advice*. If two topics need the same name, try a different topic count and see whether the resulting topics are easier to tell apart.

In [ ]:
TOPIC_NAMES = {}  # Example: {"T1": "Storage advice", "T2": "Recipe ideas"}
labels = [TOPIC_NAMES.get(topic_id, f"{topic_id} · {next(iter(topic_terms[i]), 'no terms')}")
          for i, topic_id in enumerate(topic_ids)]
best_topic = [labels[row.argmax()] if row.any() else "Unassigned" for row in shares]
profiles = articles[["article_id", "title", "url", "category"]].copy()
profiles["strongest_topic"] = best_topic
for i, topic_id in enumerate(topic_ids):
    profiles[topic_id] = (shares[:, i] * 100).round(1)
topic_dictionary = pd.DataFrame([
    {"topic_id": topic_id, "topic": label, "phrase": term, "weight": weight}
    for topic_id, label, frequencies in zip(topic_ids, labels, topic_terms)
    for term, weight in frequencies.items()
])
display(profiles)

## 3. Explore article clusters and links between topics

Clustering groups **articles** with a similar mix of topics. The n-grams helped find the topics in the previous step. The clustering uses the topic shares. Set `CLUSTERS` to 2 or 3 to try a grouping, then compare its silhouette score and the articles in each group. Scores near 1 suggest clearer groups in this sample. Scores near zero suggest overlap. A negative score suggests some articles may fit another group better.

The 3D map shows articles as points. Its axes are three directions of variation found by PCA. Two articles that appear close have broadly similar topic profiles in the displayed dimensions. The map may omit some variation. If there are fewer than three usable directions, unused axes stay at zero.

The separate 3D topic map draws a link when both topics account for at least 20% of the modelled weight in the same article. Its nodes sit on a circle for readability. Use the lines and the table to judge which topics appear together.

In [ ]:
CLUSTERS = 0  # Set to 2 or 3 to explore a grouping.
LINK_SHARE = 0.20
valid = shares.sum(axis=1) > 0
active = shares[valid]
if len(active) < 2:
    raise ValueError("Too few articles have usable topic profiles. Try fewer topics.")
dimensions = min(3, active.shape[0], active.shape[1])
pca = PCA(n_components=dimensions)
locations = pca.fit_transform(active)
xyz = np.pad(locations, ((0, 0), (0, 3-dimensions)))
profiles["cluster"] = "Unassigned"
profiles.loc[valid, "cluster"] = "Clustering off"
if CLUSTERS:
    if not 2 <= CLUSTERS < len(active):
        raise ValueError("Choose at least two clusters and fewer clusters than usable articles.")
    assignments = AgglomerativeClustering(n_clusters=CLUSTERS).fit_predict(active)
    print(f"Mean silhouette score: {silhouette_score(active, assignments):.2f}")
    profiles.loc[valid, "cluster"] = [f"Cluster {number+1}" for number in assignments]
    display(profiles[["article_id", "title", "strongest_topic", "cluster"]])

view = profiles.loc[valid].copy()
view[["x", "y", "z"]] = xyz
colour = "cluster" if CLUSTERS else "strongest_topic"
map_figure = go.Figure()
for group, rows in view.groupby(colour):
    hover = [f"{row.article_id} · {row.title}<br>" + "<br>".join(
        f"{label}: {getattr(row, topic_id):.1f}%" for label, topic_id in zip(labels, topic_ids)
    ) for row in rows.itertuples()]
    map_figure.add_scatter3d(x=rows.x, y=rows.y, z=rows.z, mode="markers",
                             name=group, text=hover, hoverinfo="text")
explained = np.nan_to_num(pca.explained_variance_ratio_).sum()
map_figure.update_layout(title=f"Articles by topic mix ({explained:.0%} of variation shown)",
    scene=dict(xaxis_title="PC 1", yaxis_title="PC 2", zaxis_title="PC 3"))
map_figure.show()

angles = np.arange(k) * 2 * np.pi / k
nodes = np.column_stack([np.cos(angles), np.sin(angles), shares.mean(axis=0)])
links, network = [], go.Figure()
for a, b in combinations(range(k), 2):
    together = (shares[:, a] >= LINK_SHARE) & (shares[:, b] >= LINK_SHARE)
    count = int(together.sum())
    links.append({"topic_a": labels[a], "topic_b": labels[b], "articles": count,
                  "article_ids": ", ".join(articles.loc[together, "article_id"])})
    if count:
        x, y, z = nodes[[a, b]].T
        network.add_scatter3d(x=x, y=y, z=z, mode="lines", showlegend=False,
            line=dict(width=min(12, 2 + count), color="#338378"),
            text=[f"{labels[a]} and {labels[b]} · {count} articles"] * 2, hoverinfo="text")
network.add_scatter3d(x=nodes[:, 0], y=nodes[:, 1], z=nodes[:, 2],
    mode="markers+text", text=labels, textposition="top center", showlegend=False,
    marker=dict(size=8, color="#155b50"))
network.update_layout(title=f"Topic links at a {LINK_SHARE:.0%} share per topic",
    scene=dict(xaxis=dict(visible=False), yaxis=dict(visible=False),
               zaxis=dict(title="Mean topic share", tickformat=".0%")))
network.show()
link_table = pd.DataFrame(links).sort_values("articles", ascending=False)
display(link_table)

## 4. Compare the blog with customer reviews

Start from your review themes and write the question a customer needs answered. Replace the illustrative examples below or upload a CSV with `theme` and `question` columns. The notebook retrieves up to three passages from different articles for each question. It uses shared words and phrases, so check for relevant articles that use different wording.

For each question, record one verdict: **Answered directly**, **Partly answered** or **No answer found in the sample**. Cite the article and a short passage that supports your judgement. If there is a gap, propose a piece of content or a change to an existing article, then name the customer behaviour you would measure.

In [ ]:
UPLOAD_REVIEW_THEMES = False
review_questions = pd.DataFrame([
    {"theme": "Recipe ideas", "question": "What can I cook with the cabbage in my box?"},
    {"theme": "Storage", "question": "How do I store fruit and veg so they last longer?"},
    {"theme": "Delivery changes", "question": "How can I change my delivery day?"}
])

if UPLOAD_REVIEW_THEMES:
    _, payload = read_upload()
    review_questions = pd.read_csv(BytesIO(payload), keep_default_na=False)
    if not {"theme", "question"}.issubset(review_questions.columns):
        raise ValueError("Use a CSV with theme and question columns.")
    review_questions = review_questions[["theme", "question"]]
review_questions = review_questions.reset_index(drop=True)

def evidence_passages(text):
    parts = [part.strip() for part in re.split(r"\n\s*\n", text) if part.strip()]
    for index, part in enumerate(parts):
        if len(part.split()) >= 12:
            yield part
        elif index + 1 < len(parts):
            yield f"{part} {parts[index+1]}"

passages = pd.DataFrame([
    {"article_id": row.article_id, "title": row.title, "url": row.url, "passage": paragraph}
    for row in articles.itertuples()
    for paragraph in evidence_passages(row.text)
])
# Use a separate search vocabulary so rare terms such as an ingredient can still match.
search_vectoriser = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
passage_matrix = search_vectoriser.fit_transform(passages["passage"])
passages["article_topic"] = passages["article_id"].map(
    profiles.set_index("article_id")["strongest_topic"]
)
audit_rows = []
candidate_rows = []
for i, row in review_questions.iterrows():
    question_id = f"Q{i+1:02d}"
    query = search_vectoriser.transform([str(row["question"])])
    scores = cosine_similarity(query, passage_matrix).ravel()
    hits = passages.assign(similarity=scores).query("similarity > 0")
    hits = (hits.sort_values("similarity", ascending=False)
            .drop_duplicates("article_id").head(3))
    hits = hits.assign(question_id=question_id)
    candidate_rows.append(hits)
    print(f"{question_id} | {row['theme']} | {row['question']}")
    for hit in hits.itertuples():
        print(f"  {hit.article_id} | {hit.title} | {hit.article_topic} | {hit.similarity:.2f} | {hit.passage[:450]}")
    if hits.empty:
        print("  No shared words or phrases found. Check the article list manually.")
    audit_rows.append({"question_id": question_id, **row.to_dict(),
        "candidate_article_ids": ", ".join(hits["article_id"]),
        "candidate_blog_topics": ", ".join(dict.fromkeys(hits["article_topic"])),
        "verdict": "", "evidence_article_id": "", "evidence": "",
        "content_action": "", "measure": ""})
audit = pd.DataFrame(audit_rows)
candidates = pd.concat(candidate_rows, ignore_index=True) if candidate_rows else passages.iloc[:0]
display(audit)

## 5. Categorise a new article

Paste an article or upload a `.txt`, `.md` or `.docx` file. The notebook applies the vocabulary and topic model from the existing articles, then gives you the topic mix and the three closest examples by cosine similarity of their TF-IDF text. If the closest examples provide a clear matching editorial category, the notebook suggests it. Otherwise, it can suggest a discovered topic or flag the draft for review. The similarity and topic-share thresholds below are classroom starting points. Try a closely related draft and a draft about a completely different subject to see how they behave.

In [ ]:
UPLOAD_DRAFT = False
DRAFT_TEXT = ""  # Paste the complete article here, using triple quotes for several paragraphs.
MIN_SIMILARITY = 0.20
MIN_TOPIC_SHARE = 0.45

def categorise_draft(draft):
    draft_matrix = vectoriser.transform([draft])
    mix = topic_shares(model.transform(draft_matrix))[0]
    display(pd.DataFrame({"topic": labels, "share (%)": (mix * 100).round(1)})
            .sort_values("share (%)", ascending=False))
    similarities = cosine_similarity(draft_matrix, matrix).ravel()
    closest = articles[["article_id", "title", "url", "category"]].assign(similarity=similarities)
    closest = closest.sort_values("similarity", ascending=False).query("similarity > 0").head(3)
    display(closest)
    if closest.empty or closest.iloc[0]["similarity"] < MIN_SIMILARITY or not mix.any():
        print("Manual review | weak match to the first-page sample.")
        return
    eligible = closest.query("similarity >= @MIN_SIMILARITY and category != ''")
    votes = eligible.groupby("category")["similarity"].sum().sort_values(ascending=False)
    if len(votes) and votes.iloc[0] > votes.sum() / 2:
        print(f"Suggested editorial category | {votes.index[0]}")
    elif mix.max() >= MIN_TOPIC_SHARE:
        print(f"Suggested discovered topic | {labels[mix.argmax()]}")
    else:
        print("Manual review | the draft spans several topics or categories.")

if UPLOAD_DRAFT:
    filename, payload = read_upload()
    suffix = Path(filename).suffix.lower()
    if suffix == ".docx":
        DRAFT_TEXT = "\n\n".join(p.text for p in Document(BytesIO(payload)).paragraphs)
    elif suffix in {".txt", ".md"}:
        DRAFT_TEXT = payload.decode("utf-8-sig")
    else:
        raise ValueError("Choose a .txt, .md or .docx file.")
if DRAFT_TEXT.strip():
    categorise_draft(DRAFT_TEXT)
else:
    print("Paste draft text or set UPLOAD_DRAFT to True, then run this cell again.")

## 6. Download your work

Run the cell to download the collected article text, topic profiles, topic links, review coverage worksheet, candidate evidence and both interactive maps as a ZIP. Open the review coverage CSV in a spreadsheet to complete your verdicts and plan the next content experiment.

In [ ]:
from google.colab import files

tables = {
    "article_topic_profiles.csv": profiles,
    "topic_dictionary.csv": topic_dictionary,
    "topic_links.csv": link_table,
    "review_coverage_worksheet.csv": audit,
    "review_candidate_passages.csv": candidates,
}
for filename, table in tables.items():
    table.to_csv(filename, index=False)
map_figure.write_html("article_map.html", include_plotlyjs=True)
network.write_html("topic_links_map.html", include_plotlyjs=True)
with zipfile.ZipFile("blog_content_audit.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for filename in ["blog_articles.csv", *tables, "article_map.html", "topic_links_map.html"]:
        archive.write(filename)
files.download("blog_content_audit.zip")